In [1]:
!pip install sounddevice scipy opencv-python

In [ ]:
"""
Live Emotion Detector — CNN + SVM Fusion
-----------------------------------------
Zero librosa in the live script — all audio features computed
with pure numpy/scipy so there is no numba dependency at all.

Controls:
    SPACE  — force an immediate prediction
    Q      — quit

Requirements:
    pip install tensorflow scikit-learn opencv-python
               numpy joblib sounddevice Pillow scipy
"""

import threading
import time
import queue
import warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import sounddevice as sd
import joblib
import tensorflow as tf
from scipy.fftpack import dct
from PIL import Image, ImageDraw, ImageFont

tf.keras.backend.clear_session()

# ─── CONFIG ───────────────────────────────────────────────────────────────────

EMOTIONS      = ["angry", "happy", "sad"]
IMG_SIZE      = (48, 48)
SAMPLE_RATE   = 22050
RECORD_SECS   = 3
N_MFCC        = 13
AUTO_INTERVAL = 0        # seconds between auto-predictions (0 = manual only)

EM_COLOR_PIL = {
    "angry": (220, 70,  50),
    "happy": (90,  200, 60),
    "sad":   (60,  100, 210),
}
PANEL_COLOR  = (18,  18,  25)
TEXT_PRIMARY = (240, 240, 255)
TEXT_DIM     = (120, 120, 145)
DIVIDER      = (50,  50,  70)

# ─── LOAD MODELS ──────────────────────────────────────────────────────────────

print("Loading models...")
cnn     = tf.keras.models.load_model("1_CNN_model.h5")
svm     = joblib.load("2_SVM_model.pkl")
scaler  = joblib.load("scaler.pkl")
weights = joblib.load("3_Combined_model.pkl")
W_CNN   = weights["w_cnn"]
W_SVM   = weights["w_svm"]
print(f"Ready.  CNN={W_CNN:.2f}  SVM={W_SVM:.2f}\n")

# ─── PURE NUMPY AUDIO FEATURES ────────────────────────────────────────────────
# Replicates exactly what librosa produced during SVM training:
# 13 MFCCs  +  pitch (Hz)  +  energy (RMS)

def preemphasis(signal, coef=0.97):
    return np.append(signal[0], signal[1:] - coef * signal[:-1])

def frame_signal(signal, frame_len, hop_len):
    n_frames = 1 + (len(signal) - frame_len) // hop_len
    idx = (np.arange(frame_len)[None, :] +
           np.arange(n_frames)[:, None] * hop_len)
    return signal[idx]

def power_spectrum(frames, nfft=512):
    windowed = frames * np.hamming(frames.shape[1])
    mag      = np.abs(np.fft.rfft(windowed, n=nfft))
    return (1.0 / nfft) * (mag ** 2)

def hz_to_mel(hz):
    return 2595.0 * np.log10(1.0 + hz / 700.0)

def mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)

def mel_filterbank(sr, nfft, n_filters=26, fmin=0, fmax=None):
    if fmax is None:
        fmax = sr / 2
    mel_min  = hz_to_mel(fmin)
    mel_max  = hz_to_mel(fmax)
    mel_pts  = np.linspace(mel_min, mel_max, n_filters + 2)
    hz_pts   = mel_to_hz(mel_pts)
    bin_pts  = np.floor((nfft + 1) * hz_pts / sr).astype(int)
    fbank    = np.zeros((n_filters, nfft // 2 + 1))
    for m in range(1, n_filters + 1):
        f_m_minus = bin_pts[m - 1]
        f_m       = bin_pts[m]
        f_m_plus  = bin_pts[m + 1]
        for k in range(f_m_minus, f_m):
            if f_m != f_m_minus:
                fbank[m-1, k] = (k - f_m_minus) / (f_m - f_m_minus)
        for k in range(f_m, f_m_plus):
            if f_m_plus != f_m:
                fbank[m-1, k] = (f_m_plus - k) / (f_m_plus - f_m)
    return fbank

def compute_mfcc(audio, sr=SAMPLE_RATE, n_mfcc=13,
                 frame_len=512, hop_len=256, nfft=512, n_filters=26):
    audio   = preemphasis(audio)
    frames  = frame_signal(audio, frame_len, hop_len)
    ps      = power_spectrum(frames, nfft)
    fbank   = mel_filterbank(sr, nfft, n_filters)
    filter_energies = np.dot(ps, fbank.T)
    filter_energies = np.where(filter_energies == 0,
                               np.finfo(float).eps, filter_energies)
    log_fbank = np.log(filter_energies)
    mfcc      = dct(log_fbank, type=2, axis=1, norm='ortho')[:, :n_mfcc]
    return np.mean(mfcc, axis=0)   # (n_mfcc,)

def compute_pitch(audio, sr=SAMPLE_RATE):
    frame_len = 2048
    hop       = 512
    lag_min   = max(1, int(sr / 2093))   # C7 ~2093 Hz
    lag_max   = int(sr / 65)             # C2 ~65 Hz
    pitches   = []
    for s in range(0, len(audio) - frame_len, hop):
        f    = audio[s:s+frame_len] * np.hanning(frame_len)
        corr = np.correlate(f, f, mode="full")[frame_len-1:]
        seg  = corr[lag_min:lag_max]
        if seg.max() > 0:
            pitches.append(sr / (np.argmax(seg) + lag_min))
    return float(np.mean(pitches)) if pitches else 0.0

def compute_rms(audio):
    return float(np.sqrt(np.mean(audio ** 2)))

def extract_features(audio):
    target = int(SAMPLE_RATE * RECORD_SECS)
    if len(audio) < target:
        audio = np.pad(audio, (0, target - len(audio)))
    audio  = audio[:target].astype("float32")
    mfccs  = compute_mfcc(audio)
    pitch  = compute_pitch(audio)
    energy = compute_rms(audio)
    return np.append(mfccs, [pitch, energy])   # (15,)

# ─── PREDICTION ───────────────────────────────────────────────────────────────

def predict_from_frame_and_audio(frame_bgr, audio):
    # CNN — image
    rgb       = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    img       = cv2.resize(rgb, IMG_SIZE).astype("float32") / 255.0
    cnn_probs = cnn.predict(np.expand_dims(img, 0), verbose=0)[0]

    # SVM — audio
    feats     = extract_features(audio)
    scaled    = scaler.transform([feats])
    raw       = svm.predict_proba(scaled)[0]
    order     = list(svm.classes_)
    svm_probs = np.array([raw[order.index(em)] for em in EMOTIONS])

    combined  = 1 * cnn_probs + 0 * svm_probs
    return {
        "prediction": EMOTIONS[np.argmax(combined)],
        "confidence": float(np.max(combined)),
        "cnn_probs":  cnn_probs,
        "svm_probs":  svm_probs,
        "combined":   combined,
    }

# ─── FONTS ────────────────────────────────────────────────────────────────────

def load_fonts():
    bold = [
        "C:/Windows/Fonts/arialbd.ttf",
        "C:/Windows/Fonts/calibrib.ttf",
        "C:/Windows/Fonts/segoeuib.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/System/Library/Fonts/Helvetica.ttc",
    ]
    reg = [
        "C:/Windows/Fonts/arial.ttf",
        "C:/Windows/Fonts/calibri.ttf",
        "C:/Windows/Fonts/segoeui.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/System/Library/Fonts/Helvetica.ttc",
    ]
    def tf(paths, size):
        for p in paths:
            try: return ImageFont.truetype(p, size)
            except: pass
        return ImageFont.load_default()
    return {
        "title":   tf(bold, 18),
        "emotion": tf(bold, 58),
        "conf":    tf(bold, 21),
        "label":   tf(bold, 16),
        "small":   tf(reg,  14),
        "status":  tf(bold, 17),
    }

FONTS = load_fonts()

# ─── DRAWING ──────────────────────────────────────────────────────────────────

def pil_text(draw, xy, text, font, color):
    x, y = xy
    draw.text((x+1, y+1), text, font=font, fill=(0,0,0,180))
    draw.text((x,   y),   text, font=font, fill=color)

def draw_panel(pil_img, result, recording, countdown, last_time):
    W, H  = pil_img.size
    pw    = 330
    cam_w = W - pw
    draw  = ImageDraw.Draw(pil_img, "RGBA")

    draw.rectangle([(cam_w, 0), (W, H)], fill=(*PANEL_COLOR, 235))
    px, py = cam_w + 18, 18

    pil_text(draw, (px, py), "EMOTION DETECTOR", FONTS["title"], TEXT_DIM)
    py += 30
    draw.line([(px, py), (W-14, py)], fill=DIVIDER, width=1)
    py += 18

    if result:
        em       = result["prediction"]
        conf     = result["confidence"]
        ec       = EM_COLOR_PIL[em]

        pil_text(draw, (px, py), em.upper(), FONTS["emotion"], ec)
        py += 68

        pil_text(draw, (px, py), f"Confidence  {conf*100:.0f}%",
                 FONTS["conf"], TEXT_PRIMARY)
        py += 28
        bw = pw - 36
        draw.rounded_rectangle([(px, py), (px+bw, py+16)],
                                radius=8, fill=(38,38,55))
        fb = int(bw * conf)
        if fb > 8:
            draw.rounded_rectangle([(px, py), (px+fb, py+16)],
                                    radius=8, fill=ec)
        py += 30

        draw.line([(px, py), (W-14, py)], fill=DIVIDER, width=1)
        py += 14

        for source, probs in [("CNN  (image)", result["cnn_probs"]),
                               ("SVM  (audio)", result["svm_probs"]),
                               ("FUSED",        result["combined"])]:
            pil_text(draw, (px, py), source, FONTS["label"], TEXT_DIM)
            py += 22
            for i, emotion in enumerate(EMOTIONS):
                ecol = EM_COLOR_PIL[emotion]
                pil_text(draw, (px, py), emotion[:3].upper(), FONTS["small"], ecol)
                bx = px + 44; nbw = pw - 110
                draw.rounded_rectangle([(bx, py), (bx+nbw, py+13)],
                                        radius=6, fill=(38,38,52))
                fb2 = int(nbw * np.clip(probs[i], 0, 1))
                if fb2 > 6:
                    draw.rounded_rectangle([(bx, py), (bx+fb2, py+13)],
                                            radius=6, fill=ecol)
                pil_text(draw, (bx+nbw+8, py),
                         f"{probs[i]*100:.0f}%", FONTS["small"], TEXT_PRIMARY)
                py += 19
            py += 10
    else:
        py += 14
        pil_text(draw, (px, py), "Press SPACE", FONTS["conf"], TEXT_DIM)
        py += 34
        pil_text(draw, (px, py), "to predict",  FONTS["conf"], TEXT_DIM)

    # Status bar
    draw.line([(px, H-68), (W-14, H-68)], fill=DIVIDER, width=1)
    if recording:
        dot = (220, 60, 60)
        draw.ellipse([(px, H-48), (px+14, H-34)], fill=dot)
        pil_text(draw, (px+20, H-50),
                 f"Recording...  {countdown:.1f}s", FONTS["status"], dot)
    else:
        if AUTO_INTERVAL > 0 and last_time is not None:
            remaining = max(0, AUTO_INTERVAL - (time.time() - last_time))
            pil_text(draw, (px, H-50),
                     f"Next in {remaining:.0f}s  |  SPACE = now",
                     FONTS["small"], TEXT_DIM)
        else:
            pil_text(draw, (px, H-50),
                     "SPACE = predict   Q = quit",
                     FONTS["small"], TEXT_DIM)

    pil_text(draw, (px, H-22),
             f"w_cnn={W_CNN:.2f}  w_svm={W_SVM:.2f}",
             FONTS["small"], (55,55,75))

    if result:
        draw.rectangle([(0,0),(cam_w-1, H-1)],
                        outline=EM_COLOR_PIL[result["prediction"]], width=5)
    return pil_img

# ─── AUDIO ────────────────────────────────────────────────────────────────────

def record_audio():
    audio = sd.rec(int(RECORD_SECS * SAMPLE_RATE),
                   samplerate=SAMPLE_RATE, channels=1, dtype="float32")
    sd.wait()
    return audio.flatten()

# ─── MAIN ─────────────────────────────────────────────────────────────────────

def main():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("ERROR: Cannot open webcam.")
        return
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    result    = None
    recording = False
    countdown = 0.0
    result_q  = queue.Queue()
    last_pred = None
    t_start   = None

    cv2.namedWindow("Emotion Detector", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Emotion Detector", 1100, 680)

    print("  ╔══════════════════════════════╗")
    print("  ║   Live Emotion Detector      ║")
    print(f"  ║   Auto-predict every {AUTO_INTERVAL}s      ║")
    print("  ║   SPACE = now  |  Q = quit   ║")
    print("  ╚══════════════════════════════╝\n")

    def run_prediction(snapshot):
        nonlocal recording, countdown
        try:
            audio = record_audio()
            res   = predict_from_frame_and_audio(snapshot, audio)
            result_q.put(res)
            print(f"  → {res['prediction'].upper()} "
                  f"({res['confidence']*100:.1f}%) | "
                  f"CNN {[f'{p*100:.0f}%' for p in res['cnn_probs']]} | "
                  f"SVM {[f'{p*100:.0f}%' for p in res['svm_probs']]}")
        except Exception as e:
            print(f"  [error] {e}")
        finally:
            recording = False
            countdown = 0.0

    def trigger(frame):
        nonlocal recording, last_pred, t_start
        if recording:
            return
        recording = True
        last_pred = time.time()
        t_start   = time.time()
        threading.Thread(target=run_prediction,
                         args=(frame.copy(),), daemon=True).start()

    while True:
        ret, frame = cap.read()
        if not ret:
            print("ERROR: Lost camera.")
            break

        if recording and t_start is not None:
            countdown = max(0.0, RECORD_SECS - (time.time() - t_start))

        # Auto-trigger
        if (AUTO_INTERVAL > 0 and not recording and
                (last_pred is None or
                 time.time() - last_pred >= AUTO_INTERVAL)):
            trigger(frame)

        try:
            result = result_q.get_nowait()
        except queue.Empty:
            pass

        pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        pil_img = draw_panel(pil_img, result, recording, countdown, last_pred)
        display = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

        cv2.imshow("Emotion Detector", display)
        key = cv2.waitKey(30) & 0xFF
        if key == ord("q") or key == 27:
            break
        elif key == ord(" "):
            trigger(frame)

    cap.release()
    cv2.destroyAllWindows()
    print("\nSession ended.")

if __name__ == "__main__":
    main()

Loading models...


Ready.  CNN=0.52  SVM=0.48

  ╔══════════════════════════════╗
  ║   Live Emotion Detector      ║
  ║   Auto-predict every 0s      ║
  ║   SPACE = now  |  Q = quit   ║
  ╚══════════════════════════════╝

  → HAPPY (79.8%) | CNN ['14%', '80%', '7%'] | SVM ['78%', '14%', '9%']
